In [1]:
import pandas as pd

SCMP_PATH = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_scmp.xlsx'
scmp = pd.read_excel(SCMP_PATH)
print(f'SCMP snippets : {len(scmp)}')
print(f'Distinct articles : {scmp["article_id"].nunique()}')
print(f'Columns           : {list(scmp.columns)}')
scmp.head(3)


SCMP snippets : 1559
Distinct articles : 62
Columns           : ['article_id', 'title', 'outlet', 'date', 'snippet', 'word_count', 'china', 'indonesia', 'nickel']


,article_id,title,outlet,date,snippet,word_count,china,indonesia,nickel
0,urn:contentItem:60HT-VXT1-JC8V-135X-00000-00,Protesting Indonesian students write to Chines...,"South China Morning Post.com, EDT, 1037words",2020-08-06,Protesting Indonesian students write to Chines...,120,1,1,1
1,urn:contentItem:60HT-VXT1-JC8V-135X-00000-00,Protesting Indonesian students write to Chines...,"South China Morning Post.com, EDT, 1037words",2020-08-06,"Since March, the students have been holding de...",62,0,0,1
2,urn:contentItem:60HT-VXT1-JC8V-135X-00000-00,Protesting Indonesian students write to Chines...,"South China Morning Post.com, EDT, 1037words",2020-08-06,The protesters say the Chinese arrivals are no...,63,1,0,0


In [2]:
# below is prepared in NB1 

import json as _json

NER_PATH = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\ner_actor_codebook.json'
with open(NER_PATH, 'r', encoding='utf-8') as f:
    ner_records = _json.load(f)

print(f'Actor categories loaded: {len(ner_records)}')
for rec in ner_records:
    print(f"  {rec['actor_label']}: {len(rec['named_entities'])} entities, {len(rec['trigger_phrases'])} trigger phrases")


Actor categories loaded: 13
  Chinese DFIs (banks): 7 entities, 16 trigger phrases
  Chinese firms/SOEs: 10 entities, 13 trigger phrases
  Chinese government: 8 entities, 11 trigger phrases
  Chinese worker: 6 entities, 9 trigger phrases
  Indonesian government: central government/president: 7 entities, 11 trigger phrases
  Indonesian government: local government (provincial/regional officials): 6 entities, 10 trigger phrases
  Indonesian firms and local elites: 10 entities, 11 trigger phrases
  Indonesian local civil society: 8 entities, 11 trigger phrases
  Indonesian local community & workers: 6 entities, 13 trigger phrases
  International government (EU/US/others): 8 entities, 16 trigger phrases
  International companies: 10 entities, 12 trigger phrases
  International NGOs/watchdog: 6 entities, 7 trigger phrases
  *Others: 4 entities, 0 trigger phrases


In [3]:
import re

def make_pattern(term):
    """Build a regex pattern that matches term and its common plural/suffix forms."""
    t = term.lower()
    # y → ies (community → communities, company → companies)
    if t.endswith('y') and len(t) > 2 and t[-2] not in 'aeiou':
        core = re.escape(t[:-1])
        return r'\b' + core + r'(?:y|ies)\b'
    # s/x/ch/sh/z → es (process → processes)
    elif re.search(r'(?:s|x|ch|sh|z)$', t):
        return r'\b' + re.escape(t) + r'(?:es)?\b'
    # default: optional trailing s (worker → workers, government → governments)
    else:
        return r'\b' + re.escape(t) + r's?\b'

# these terms are handled by extract_context_words instead
CONTEXT_ONLY = {'market', 'environment'}

def extract_named_entities(text):
    """Match named entities -> 'Entity (Category)' format."""
    found = []
    text_lower = str(text).lower()
    for rec in ner_records:
        label = rec['actor_label']
        for ent in rec['named_entities']:
            if ent.lower() in CONTEXT_ONLY:
                continue  # skip — captured with context instead
            if re.search(make_pattern(ent), text_lower):
                found.append(f'{ent} ({label})')
    return '; '.join(found) if found else ''

def extract_trigger_signals(text):
    """Match trigger phrases -> '"phrase" -> Category' format."""
    found = []
    text_lower = str(text).lower()
    seen = set()
    for rec in ner_records:
        label = rec['actor_label']
        for phrase in rec['trigger_phrases']:
            if re.search(make_pattern(phrase), text_lower):
                entry = f'"{phrase}" -> {label}'
                if entry not in seen:
                    seen.add(entry)
                    found.append(entry)
    return '; '.join(found) if found else ''


def extract_context_words(text, keywords=('market', 'environment')):
    """For each keyword found, capture the word before and after it as context."""
    words = re.findall(r'\b\w+\b', str(text).lower())
    found = []
    for i, word in enumerate(words):
        if word in keywords:
            before = words[i - 1] if i > 0 else ''
            after  = words[i + 1] if i < len(words) - 1 else ''
            context = ' '.join(filter(None, [before, word, after]))
            found.append(f'{context} [context]')
    return '; '.join(found) if found else ''

scmp = scmp.copy()
scmp['actors_mentioned'] = scmp['snippet'].apply(extract_named_entities)
scmp['actor_signals']    = scmp['snippet'].apply(extract_trigger_signals)

# append word-context hits for 'market' and 'environment'
scmp['actors_mentioned'] = scmp.apply(
    lambda r: '; '.join(filter(None, [r['actors_mentioned'],
                                      extract_context_words(r['snippet'])])),
    axis=1
)

n_ent  = (scmp['actors_mentioned'] != '').sum()
n_sig  = (scmp['actor_signals']    != '').sum()
n_both = ((scmp['actors_mentioned'] != '') & (scmp['actor_signals'] != '')).sum()
print(f'Snippets matched by named entity    : {n_ent} / {len(scmp)} ({n_ent/len(scmp)*100:.1f}%)')
print(f'Snippets matched by trigger phrase  : {n_sig} / {len(scmp)} ({n_sig/len(scmp)*100:.1f}%)')
print(f'Snippets matched by both            : {n_both}')
scmp[scmp['actors_mentioned'] != ''][['snippet','actors_mentioned','actor_signals']].head(3)


KeyboardInterrupt: 

In [ ]:
# ── Plural matching sanity check ─────────────────────────────────────────────
import re

# pairs: (term from codebook, plural form that must also match)
TEST_CASES = [
    # y → ies
    ('community',    'communities'),
    ('company',      'companies'),
    ('authority',    'authorities'),
    ('ministry',     'ministries'),
    # default s
    ('government',   'governments'),
    ('worker',       'workers'),
    ('investor',     'investors'),
    ('activist',     'activists'),
    ('contractor',   'contractors'),
    ('organization', 'organizations'),
    # singular still matches
    ('government',   'government'),
    ('community',    'community'),
    # named entities — plural should not break match
    ('Tesla',        'Tesla'),
    ('Greenpeace',   'Greenpeace'),
]

print(f'{"Term":<20} {"Test string":<20} {"Pattern":<35} {"Match"}')
print('-' * 85)
all_pass = True
for term, test_str in TEST_CASES:
    pattern = make_pattern(term)
    match   = bool(re.search(pattern, test_str.lower()))
    status  = 'PASS' if match else 'FAIL'
    if not match: all_pass = False
    print(f'{term:<20} {test_str:<20} {pattern:<35} {status}')

print()
print('All tests passed.' if all_pass else 'Some tests FAILED — review make_pattern().')

# also spot-check against real trigger phrases from the codebook
print('\n--- Live codebook terms ---')
sample_phrases = [(rec['actor_label'], p)
                  for rec in ner_records
                  for p in rec['trigger_phrases'][:2]]
test_sentences = [
    'local governments have issued permits',
    'mining communities were displaced',
    'Chinese companies and investors',
    'provincial authorities cracked down',
    'construction workers from China',
]
for sentence in test_sentences:
    hits = []
    for rec in ner_records:
        for p in rec['trigger_phrases']:
            if re.search(make_pattern(p), sentence.lower()):
                hits.append(f'{p} ({rec["actor_label"]})')
    print(f'  "{sentence}"')
    print(f'    -> {hits if hits else "no match"}')


Term                 Test string          Pattern                             Match
-------------------------------------------------------------------------------------
community            communities          \bcommunit(?:y|ies)\b               PASS
company              companies            \bcompan(?:y|ies)\b                 PASS
authority            authorities          \bauthorit(?:y|ies)\b               PASS
ministry             ministries           \bministr(?:y|ies)\b                PASS
government           governments          \bgovernments?\b                    PASS
worker               workers              \bworkers?\b                        PASS
investor             investors            \binvestors?\b                      PASS
activist             activists            \bactivists?\b                      PASS
contractor           contractors          \bcontractors?\b                    PASS
organization         organizations        \borganizations?\b                  PASS


In [ ]:
from collections import Counter

# ── named entity frequency ───────────────────────────────────────────────────
cat_counts    = Counter()
entity_counts = Counter()
for cell in scmp['actors_mentioned']:
    if not cell:
        continue
    for match in cell.split('; '):
        match = match.strip()
        if not match:
            continue
        entity_counts[match] += 1
        m = re.search(r'\((.+?)\)$', match)
        if m:
            cat_counts[m.group(1)] += 1

# ── trigger phrase frequency ─────────────────────────────────────────────────
sig_cat_counts    = Counter()
sig_phrase_counts = Counter()
for cell in scmp['actor_signals']:
    if not cell:
        continue
    for match in cell.split('; '):
        match = match.strip()
        if not match:
            continue
        sig_phrase_counts[match] += 1
        m = re.search(r'->\s*(.+)$', match)
        if m:
            sig_cat_counts[m.group(1).strip()] += 1

# ── combined category table ──────────────────────────────────────────────────
all_cats = set(cat_counts) | set(sig_cat_counts)
combined = pd.DataFrame([
    {'actor_category'   : c,
     'entity_mentions'  : cat_counts.get(c, 0),
     'trigger_mentions' : sig_cat_counts.get(c, 0)}
    for c in all_cats
]).sort_values('entity_mentions', ascending=False).reset_index(drop=True)
combined['total'] = combined['entity_mentions'] + combined['trigger_mentions']
combined = combined.sort_values('total', ascending=False).reset_index(drop=True)

print('=== Mentions per actor category ===')
print(combined.to_string(index=False))

# ── top entities + top trigger phrases ───────────────────────────────────────
ent_df = (pd.DataFrame.from_dict(entity_counts, orient='index', columns=['mentions'])
            .sort_values('mentions', ascending=False).head(20)
            .reset_index().rename(columns={'index': 'entity'}))
sig_df = (pd.DataFrame.from_dict(sig_phrase_counts, orient='index', columns=['mentions'])
            .sort_values('mentions', ascending=False).head(20)
            .reset_index().rename(columns={'index': 'trigger_signal'}))

print('\n=== Top 20 named entity mentions ===')
print(ent_df.to_string(index=False))
print('\n=== Top 20 trigger phrase signals ===')
print(sig_df.to_string(index=False))


# save top 100 snippets by number of entity mentions
scmp['n_entity_mentions'] = scmp['actors_mentioned'].apply(
    lambda x: len([e for e in x.split(';') if e.strip()]) if x else 0
)
top100 = scmp.nlargest(100, 'n_entity_mentions')

SCMP_OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\100_snippets_scmp_NER.xlsx'
top100.to_excel(SCMP_OUT, index=False)
print(f'Top 100 snippets by entity mentions saved: {SCMP_OUT}')
print(f'Entity mention counts range: {top100["n_entity_mentions"].min()} – {top100["n_entity_mentions"].max()}')

=== Mentions per actor category ===
                                                         actor_category  entity_mentions  trigger_mentions  total
                    Indonesian government: central government/president              141               439    580
                                                     Chinese firms/SOEs               87               147    234
                                                     Chinese government              101               109    210
                                International government (EU/US/others)               10               179    189
                                                International companies               85                39    124
                                         Indonesian local civil society               15                77     92
                                   Indonesian local community & workers               23                48     71
                                                    

In [ ]:
# ── Asia Times NER extraction ────────────────────────────────────────────────
AT_PATH = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_asia_times.xlsx'
asia_times = pd.read_excel(AT_PATH)

asia_times = asia_times.copy()
asia_times['actors_mentioned'] = asia_times['snippet'].apply(extract_named_entities)
asia_times['actor_signals']    = asia_times['snippet'].apply(extract_trigger_signals)
asia_times['actors_mentioned'] = asia_times.apply(
    lambda r: '; '.join(filter(None, [r['actors_mentioned'],
                                      extract_context_words(r['snippet'])])), axis=1
)

asia_times['n_entity_mentions'] = asia_times['actors_mentioned'].apply(
    lambda x: len([e for e in x.split(';') if e.strip()]) if x else 0
)
top100_at = asia_times.nlargest(100, 'n_entity_mentions')

AT_OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\100_snippets_asia_times_NER.xlsx'
top100_at.to_excel(AT_OUT, index=False)
print(f'Saved: {AT_OUT}')
print(f'Entity mention counts range: {top100_at["n_entity_mentions"].min()} – {top100_at["n_entity_mentions"].max()}')


Saved: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\100_snippets_asia_times_NER.xlsx
Entity mention counts range: 1 – 4
